# Optimizer — self-recovery on a small polymer

**The honesty gate.** Invent a known θ, simulate it, treat that map as
"experimental", fit θ back, and check recovery. If this fails, nothing
downstream can be trusted.

Runs on a small polymer (N≈300) because a full optimizer iteration there is
minutes rather than an hour. Discovering a convergence problem after a week of
chr10 compute is exactly how Fi-chrom's six-day non-convergence happened.

**Why a synthetic target and not a real chr10 sub-region:** a sub-region's
measured map includes contacts with the rest of the chromosome and the rest of
the nucleus, which an isolated simulation does not reproduce. Fitting it would
silently absorb that boundary mismatch into Λ — the fit-vs-simulate environment
mismatch the retraining rule warns about. A synthetic target is generated in
exactly the environment it is fitted in.

**What to watch**
1. Does λ come back? (k=1, the cheapest case)
2. Is the recovery error at the replica-noise floor, or above it?
3. Does the adaptive budget behave, or run to the ceiling?

## 0. Setup

In [ ]:
from google.colab import drive
!git clone -q https://github.com/darinddv/chromatin_potential.git
!pip install -q "openmm[cuda12]" OpenMiChroM
import sys; sys.path.insert(0, '/content/chromatin_potential/src')
drive.mount('/content/drive')

import os, json, numpy as np
import matplotlib.pyplot as plt

from chromatin_potential.model import Model
from chromatin_potential.simulator import (
    Simulator, BackgroundStack, SaveSpec,
    observed_over_expected, saddle_strength)
from chromatin_potential import optimizer as O
print('setup OK')

In [ ]:
DATA = '/content/drive/MyDrive/uky_cheng/tecsas_ablation'
OUT  = f'{DATA}/optimizer_selfrec'
os.makedirs(OUT, exist_ok=True)

N_BEADS = 300          # small polymer: ~30-80x faster than chr10
names   = [f't{i:05d}' for i in range(N_BEADS)]
SEQ     = f'{OUT}/perbead_{N_BEADS}.txt'
with open(SEQ,'w') as fh:
    for i,n in enumerate(names): fh.write(f'{i+1} {n}\n')
print('sequence file written:', SEQ)

## 1. Timing check — run ONE replica before anything else

The whole plan rests on a small-polymer iteration being cheap. Measure it rather
than assume it. An optimizer iteration is roughly
`n_replicas × (n_steps / 3e6) × this time`.

In [ ]:
sim = Simulator(seq_file=SEQ, out_dir=OUT, platform='cuda',
                background=BackgroundStack())

C_probe = O.synthetic_coordinate(N_BEADS, k=1, n_domains=12, seed=0)
probe   = Model(C_probe, np.array([[-1.0]]), c=-0.30, names=names)

import time
t0 = time.time()
sim.run_replica('timing', 0, model=probe, n_production=50_000,
                save_spec=SaveSpec(contact_map=True, trajectory=True,
                                   interval=1000, monitor=False),
                overwrite=True, verbose=True)
dt = time.time() - t0
print(f'\n50k production steps took {dt/60:.2f} min')
print(f'-> one 4-replica iteration at 50k steps ~ {4*dt/60:.1f} min')
print(f'-> 60 iterations ~ {60*4*dt/3600:.1f} hours')
print('If this is too slow, drop N_BEADS to 200 (cost scales ~N^2).')

## 2. Define ground truth (k=1)

A smooth continuous coordinate plus a single scalar λ. This is the cheapest
possible fit — one parameter — and it is the arm that produced the headline
continuous result, so it is the right first test.

In [ ]:
C_true   = O.synthetic_coordinate(N_BEADS, k=1, n_domains=12, seed=1)
LAM_TRUE = -1.2
true_model = Model(C_true, np.array([[LAM_TRUE]]), c=-0.30, names=names)

print(f'true lambda = {LAM_TRUE}')
M = true_model.coupling_matrix()
print(f'coupling matrix: {M.shape}  mean {M.mean():+.4f}  std {M.std():.4f}')

fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].plot(C_true[:,0], lw=.9); ax[0].set_title('true coordinate C'); ax[0].set_xlabel('bead')
im = ax[1].imshow(M, cmap='RdBu_r'); ax[1].set_title('coupling matrix M')
plt.colorbar(im, ax=ax[1]); plt.tight_layout(); plt.show()

## 3. Generate the synthetic target

20 replicas at full production length — the target should be as clean as your
real experimental maps, since fitting a noisy target just fits noise.

In [ ]:
target = O.make_synthetic_target(sim, true_model, n_replicas=20,
                                 n_production=3_000_000,
                                 cond='truth', verbose=True)
np.save(f'{OUT}/target.npy', target)
print('target map:', target.shape)

### 3a. Replica-noise ceiling — what perfect recovery looks like

Correlation between two INDEPENDENT ensembles of the SAME θ. **The recovery
ceiling is not 1.0**, and judging a fit against 1.0 will make a correct fit look
like a failure. This number is also the first real identifiability quantity the
project produces.

In [ ]:
ceiling, A, B = O.replica_noise_ceiling(sim, true_model, n_replicas=10,
                                       n_production=3_000_000,
                                       cond='ceil', verbose=False)
print(f'replica-noise ceiling = {ceiling:.5f}')
print('-> a fit reaching this correlation has recovered as well as the data allows')

## 4. Fit

Start λ deliberately wrong (−0.3 vs true −1.2) so recovery is a real test, not a
short walk from the answer.

Conservative learning rate on purpose: too large a step overshoots and then trips
the plateau test, which *looks* like convergence but leaves the parameter wrong.
If the loss plateaus while still far above the noise floor, lower `lr_lambda`
rather than raising `patience`.

In [ ]:
init_model = Model(C_true.copy(), np.array([[-0.3]]), c=-0.30, names=names)

budget = O.Budget(n_steps=50_000, n_replicas=4,
                  min_steps=50_000, max_steps=400_000,
                  min_replicas=2,  max_replicas=16)

opt = O.Optimizer(sim, target, lr_lambda=0.05, fit_c=False,
                  budget=budget, out_dir=f'{OUT}/fits', tag='k1', verbose=True)
res = opt.fit(init_model, n_iter=60, patience=12)

## 5. Did it recover?

In [ ]:
rep = O.recovery_report(true_model, res.model)
rep['replica_noise_ceiling'] = ceiling

lam_fit = res.model.Lam[0,0]
print(f'true lambda    = {LAM_TRUE:+.4f}')
print(f'fitted lambda  = {lam_fit:+.4f}')
print(f'absolute error = {abs(lam_fit-LAM_TRUE):.4f}  ({abs(lam_fit-LAM_TRUE)/abs(LAM_TRUE):.1%})')
print()
print(f"M relative error = {rep['M_relative_error']:.4f}   <- PRIMARY metric")
print(f"M scale ratio    = {rep['M_scale_ratio']:.4f}   (1.0 = right magnitude)")
print(f"M correlation    = {rep['M_correlation']:.6f}   <- DEGENERATE for k=1:")
print('    M = c + lambda*C C^T, so the off-diagonal SHAPE is proportional to')
print('    C C^T for ANY lambda. Correlation reads ~1.000 even when lambda is')
print('    wrong by 8x. Ignore it here; use relative error and scale ratio.')
print()
print(f"stop reason: {res.stop_reason}")
json.dump(rep, open(f'{OUT}/recovery_report_k1.json','w'), indent=1, default=str)

### 5a. Optimization traces

In [ ]:
loss = res.trace('loss'); gn = res.trace('grad_norm')
snr  = res.trace('snr_max')
lam  = np.array([h['Lambda'][0][0] for h in res.history])
steps = res.trace('n_steps'); reps = res.trace('n_replicas')

fig, ax = plt.subplots(1, 4, figsize=(17, 3.3))
ax[0].semilogy(loss, 'o-'); ax[0].set_title('loss'); ax[0].set_xlabel('iteration')
ax[1].plot(lam, 'o-'); ax[1].axhline(LAM_TRUE, c='r', ls='--', label='true')
ax[1].set_title('lambda'); ax[1].legend(); ax[1].set_xlabel('iteration')
ax[2].semilogy(gn, 'o-'); ax[2].set_title('|gradient|'); ax[2].set_xlabel('iteration')
ax[3].plot(steps/1000, 'o-', label='ksteps'); ax[3].plot(reps, 's-', label='replicas')
ax[3].set_title('adaptive budget'); ax[3].legend(); ax[3].set_xlabel('iteration')
plt.tight_layout(); plt.show()

print('lambda trace oscillating -> lower lr_lambda.')
print('budget pinned at ceiling early -> target map too noisy, or lr too small.')

### 5b. Behavioural check — simulate the fitted model

Parameter recovery is necessary but not sufficient. The real question is whether
the *fitted* model reproduces the target's behaviour.

In [ ]:
P_fit = sim.run_ensemble('fitted', model=res.model, n_replicas=10,
                         save_spec=SaveSpec(contact_map=True, trajectory=False,
                                            monitor=False),
                         n_production=3_000_000, verbose=False)

iu = np.triu_indices(target.shape[0], 3)
r_fit = np.corrcoef(target[iu], P_fit[iu])[0,1]
print(f'fitted vs target map correlation = {r_fit:.5f}')
print(f'replica-noise ceiling            = {ceiling:.5f}')
print(f'-> {"AT the noise floor: recovered as well as the data allows" if r_fit >= ceiling-0.005 else "BELOW the ceiling: real residual error remains"}')

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
for a,(Mp,t) in zip(ax, [(target,'target (truth)'), (P_fit,'fitted'),
                         (P_fit-target,'difference')]):
    cm = 'RdBu_r' if 'diff' in t else 'Reds'
    im = a.imshow(Mp, cmap=cm); a.set_title(t); plt.colorbar(im, ax=a)
plt.tight_layout(); plt.show()

## 6. Harder case: free C (blind reduced-rank)

Fit **both** C and Λ from a random start — no biological prior at all. This is
where the gauge freedom bites, and where the Procrustes rotation magnitude
becomes the identifiability diagnostic: near-identity means the parameterization
is pinned down, a large rotation means a flat direction was found.

Optional and slower. Run it once the k=1 case passes.

In [ ]:
RUN_FREE_C = False   # flip on after k=1 passes

if RUN_FREE_C:
    free_init = Model.free(N_BEADS, k=1, seed=3, scale=0.5)
    free_init.names = names
    free_init.c = -0.30
    opt2 = O.Optimizer(sim, target, lr_lambda=0.05, lr_C=0.01,
                       budget=O.Budget(n_steps=50_000, n_replicas=4),
                       out_dir=f'{OUT}/fits', tag='freeC', verbose=True)
    res2 = opt2.fit(free_init, n_iter=80, patience=15)
    rep2 = O.recovery_report(true_model, res2.model)
    print(f"\nM relative error   : {rep2['M_relative_error']:.4f}")
    print(f"M scale ratio      : {rep2['M_scale_ratio']:.4f}")
    if 'rotation_magnitude' in rep2:
        print(f"rotation magnitude : {rep2['rotation_magnitude']:.4f}")
        print('   near 0 = parameterization pinned; large = flat direction found')
    print('\nNOTE: with free C the raw C and Lambda are only determined up to a')
    print('gauge transformation (C -> MC, Lambda -> M^-T Lambda M^-1 leaves M')
    print('unchanged). Judge recovery on M, never on raw C or Lambda.')
else:
    print('skipped — set RUN_FREE_C = True after the k=1 case passes')

## 7. What this establishes

If λ came back within the noise floor:
- the max-ent gradient and its projection onto (C, Λ) are correct,
- warm-starting and the adaptive budget work,
- **the optimizer can be trusted on real data.**

The gap between the recovered λ and truth *is* the identifiability result — it is
the precision with which a contact map determines this parameter, which is the
question the whole inference aim is built around.

**Next:** the chr10 continuous fit — fitting λ for the real PC1 coordinate, which
removes the inherited-gauge caveat from the completed `cont` result (where the
scale was chosen by hand and overshot experiment at 3.06 vs 2.76).